1. Carga de datos:

In [ ]:
import numpy as np

# Ejecutar el archivo fuente para registrar las funciones
%run ../src/retail-sales-analysis.py

# 1. Cargar datos
ruta = '../data/retail_sales_dataset.csv'
datos = cargar_datos(ruta)
datos

2. Exploración inicial:

In [ ]:
# Inspeccionar dimensiones y tipos de datos
print(f"Dimensiones del dataset: {datos.shape}")
print(f"Estructura detectada (dtype): {datos.dtype}")

3. Se lleva a cabo una traducción de los códigos resultantes de hacer 'dtype', para saber con exactitud el tipo de datos presentes en el dataset

In [ ]:
# Mapeo de nombres reales de las columnas (según el dataset de Retail de Kaggle)
categorias = [
    "Transaction ID", "Date", "Customer ID", "Gender", "Age", 
    "Product Category", "Quantity", "Price per Unit", "Total Amount"
]

print("--- DICCIONARIO DE DATOS TÉCNICO-FUNCIONAL ---")
# Encabezados de la tabla
encabezado = f"{'ID':<4} | {'Nombre Real':<18} | {'Código':<8} | {'Tipo':<12} | {'Ejemplo Real'}"
print(encabezado)
print("-" * len(encabezado))

# Registro de muestra (fila 0)
primera_fila = datos[0]

# Iterar sobre la estructura del dataset
for i, id_tec in enumerate(datos.dtype.names):

    # Obtener el código (ej: <i8)
    codigo = datos.dtype[id_tec].str
    
    # Traducir el código a un tipo legible
    if 'i' in codigo:
        tipo = "Entero"
    elif 'U' in codigo:
        tipo = "Texto"
    elif 'f' in codigo:
        tipo = "Decimal"
    else:
        tipo = "Otro"
    
    # Obtener y limpiar el dato de ejemplo
    valor = primera_fila[id_tec]
    if isinstance(valor, (bytes, np.bytes_)):
        valor = valor.decode('utf-8')
    
    # El nombre real se toma de nuestra lista manual
    columna = categorias[i] if i < len(categorias) else "Desconocido"
    
    print(f"{id_tec:<4} | {columna:<18} | {codigo:<8} | {tipo:<12} | {valor}")

print("-" * len(encabezado))

4. Chequear extremos del dataset

In [ ]:
# Visualizar primeros y últimos lugares en el dataset
print("\nPrimeros 3 registros:")
print(datos[:3])
print("\nÚltimos 3 registros:")
print(datos[-3:])

5. Chequear si existen datos faltantes

In [ ]:
# Analizar valores faltantes
print("--- VALORES FALTANTES ---")

# Recorrer la lista de nombres de columnas
for i, nombre in enumerate(datos.dtype.names):
    
    # Creamos un contador para datos faltantes
    vacia = 0
    
    # Recorrer cada fila del dataset para revisar esa columna específica
    for fila in datos:
        dato = fila[i]
        
        # Verificar si el dato es un texto vacío o un byte vacío(Numpy podría haber cargado de ambas maneras)
        if dato == '' or dato == b'': 
            vacia = vacia + 1
            
    # Imprimir el resultado de la columna actual antes de pasar a la siguiente
    print(f"Columna {nombre} (índice {i}): {vacia} valores faltantes")

7. Chequear "Falsos Nulos" (NaN, None, "null")

In [ ]:
print("--- VALORES FALSOS NULOS ---")

for i, nombre in enumerate(datos.dtype.names):
    falsos_nulos = 0
    
    for fila in datos:
        dato = fila[i]
        
        # Convertimos el dato a texto y a minúsculas para atrapar 'None', 'none', 'NULL', etc.
        dato_texto = str(dato).lower()
        
        if dato_texto == 'none' or dato_texto == 'nan' or dato_texto == 'null':
            falsos_nulos = falsos_nulos + 1
            
    print(f"Columna {nombre} (índice {i}): {falsos_nulos} nulos encontrados")

6. Chequear outliers

In [ ]:
# Extraer los montos totales de la columna 8
montos_ventas = []
for fila in datos:
    # fila[8] es el "Total Amount" (Monto Total de la Venta)
    valor_venta = float(fila[8]) 
    montos_ventas.append(valor_venta)

# Se crea un arreglo para trabajar con Numpy
ventas_array = np.array(montos_ventas)

# Encontrar los puntos de corte (Cuartiles)
q1 = np.percentile(ventas_array, 25) # El 25% de las ventas más bajas
q3 = np.percentile(ventas_array, 75) # El 75% de las ventas (donde empiezan las ventas altas)

# Calcular el "Ancho de la Venta Normal" (IQR)
rango_ventas = q3 - q1

# Establecer los límites de lo que aceptamos como "Venta Común"
limite_superior = q3 + (1.5 * rango_ventas)
limite_inferior = q1 - (1.5 * rango_ventas)

# Buscar las ventas que se salen de los límites
ventas_fuera_de_rango = []
for i in ventas_array:
    # Si la venta es mayor al límite superior o menor al inferior
    if i > limite_superior or i < limite_inferior:
        ventas_fuera_de_rango.append(i)

# Mostrar resultados
print("--- ANÁLISIS DE VENTAS TOTALES (OUTLIERS) ---")
print(f"Límite máximo para una venta normal: ${limite_superior:.2f}")
print(f"Límite mínimo para una venta normal: ${limite_inferior:.2f}")
print(f"Cantidad de ventas detectadas como atípicas: {len(ventas_fuera_de_rango)}")

if len(ventas_fuera_de_rango) > 0:
    print(f"Algunas de estas ventas fueron: {ventas_fuera_de_rango[:5]}")

### Nota: Interpretación del Límite Inferior Negativo

Al realizar el análisis de valores atípicos (outliers) mediante el método del Rango Intercuartiles (IQR), se observa que el **Límite Inferior** arroja un valor negativo. A continuación, se explica por qué esto es correcto y qué significa para nuestro análisis:

* **¿Por qué es negativo?**: El cálculo es puramente matemático ($Q1 - 1.5 \times IQR$). Si la dispersión de los precios es muy grande comparada con los valores bajos, la fórmula resta más de lo que hay, cruzando la barrera del cero.
* **Sentido de Realidad**: En el contexto de este dataset de Retail, los montos totales de venta no pueden ser menores a cero.
* **Conclusión**: Un límite inferior negativo no es un error. Simplemente nos confirma que **no existen ventas atípicas por ser "demasiado bajas"**. Todas las ventas pequeñas del dataset entran dentro del rango considerado como "normal".

7. Exporar si existen duplicados

In [ ]:
print("--- BUSCANDO REGISTROS REPETIDOS ---")

filas_vistas = []
duplicados_encontrados = []

for fila in datos:
    # Convertir la fila a una lista
    fila_lista = list(fila)
    
    if fila_lista in filas_vistas:
        duplicados_encontrados.append(fila_lista)
    else:
        filas_vistas.append(fila_lista)

print(f"Cantidad de filas duplicadas: {len(duplicados_encontrados)}")

if len(duplicados_encontrados) > 0:
    print("Se encontraron datos repetidos. Es necesario eliminarlos para no alterar las estadísticas.")

7. Exploración de datos

In [ ]:
categorias_analisis = ['Beauty', 'Clothing', 'Electronics']

# Listas para guardar los resultados
nombres_cats = []
resultados_totales = []

print("--- ANÁLISIS DE VENTAS DIARIAS POR CATEGORÍA ---")

for cat in categorias_analisis:
    # Filtrar los datos por categoría
    datos_cat = filtrar_por_categoria(datos, cat)
    
    if len(datos_cat) > 0:
        # Crear un diccionario para agrupar ventas por fecha
        ventas_por_dia = {}
        
        for fila in datos_cat:
            fecha = fila[1] # Columna 1 es la fecha
            monto = float(fila[8]) # Columna 8 es el monto
            
            # Si el día ya existe, suma monto al total del día
            if fecha in ventas_por_dia:
                ventas_por_dia[fecha] = ventas_por_dia[fecha] + monto
            # Si no existe, se agrega
            else:
                ventas_por_dia[fecha] = monto
        
        # Declarar lista con los totales de cada día
        totales_diarios = list(ventas_por_dia.values())
        
        # Calcular promedio diario
        promedio_diario = np.mean(totales_diarios)
        total_categoria = np.sum(totales_diarios)
        
        # Guardar nombre y total para comparar al final
        nombres_cats.append(cat)
        resultados_totales.append(total_categoria)
        
        print(f"\nCategoría: {cat}")
        print(f"  - Total acumulado: ${total_categoria:,.2f}")
        print(f"  - Promedio de ventas diarias: ${promedio_diario:,.2f}")
        print(f"  - Cantidad de días con ventas: {len(totales_diarios)}")

# --- IDENTIFICACIÓN DE MAYORES Y MENORES ---
print("\n--- IDENTIFICACIÓN FINAL ---")

# valores extremos basados en el total acumulado
mayor_v = max(resultados_totales)
menor_v = min(resultados_totales)

# Categorías correspondientes almax y min de ventas totales
cat_mayor = nombres_cats[resultados_totales.index(mayor_v)]
cat_menor = nombres_cats[resultados_totales.index(menor_v)]

# Resultado final
print(f"Tras el análisis, la categoría con mayores ventas es '{cat_mayor}' y la categoría con menores ventas es '{cat_menor}'.")